# Chat Messages
The `chat.py` module defines chat messages that can use an arbitrary speaker or role. It also provides a streaming chunk variant that can be combined with compatible message chunks.
# ChatMessage: `BaseMessage`
`ChatMessage` represents a message associated with an arbitrary speaker or role.
**Syntax**
```python
ChatMessage(
  self,
  content: str | list[str | dict[Any, Any]] | None = None,
  content_blocks: list[types.ContentBlock] | None = None,
  **kwargs: Any = {}
)
```
## Fields
1. `role`:`str`:= Stores the speaker or role associated with the message.
2. `type`:`Literal["chat"]`:= Stores the message type used during serialization and deserialization. Its default value is `"chat"`.


In [2]:
from langchain_core.messages import ChatMessage


message = ChatMessage(
    content="The deployment has completed successfully.", # Message content
    role="developer", # Arbitrary speaker or role
    name="Saad", # Optional message name
    id=101, # Numeric ID automatically converted to string
    additional_kwargs={
        "priority": "high" # Provider-specific information
    },
    response_metadata={
        "model_name": "example-model" # Response-related metadata
    }
)

print("message: ", message)
print("Content:", message.content) # Display message content
print("Role:", message.role) # Display arbitrary speaker or role
print("Type:", message.type) # Display serialization type
print("Name:", message.name) # Display optional message name
print("ID:", message.id) # Display string identifier
print("Additional kwargs:", message.additional_kwargs) # Display provider-specific data
print("Response metadata:", message.response_metadata) # Display response metadata

message.pretty_print() # Print a readable representation

message:  content='The deployment has completed successfully.' additional_kwargs={'priority': 'high'} response_metadata={'model_name': 'example-model'} name='Saad' id='101' role='developer'
Content: The deployment has completed successfully.
Role: developer
Type: chat
Name: Saad
ID: 101
Additional kwargs: {'priority': 'high'}
Response metadata: {'model_name': 'example-model'}
================================= Chat Message =================================
Name: Saad

The deployment has completed successfully.


# ChatMessageChunk: `ChatMessage`, `BaseMessageChunk`
`ChatMessageChunk` represents a partial chat message produced during streaming. Compatible chunks can be combined while preserving the role and merging their content and metadata.
**Syntax**
```python
ChatMessageChunk(
  self,
  content: str | list[str | dict[Any, Any]] | None = None,
  content_blocks: list[types.ContentBlock] | None = None,
  **kwargs: Any = {}
)
```
## Fields
1. `type`:`Literal["ChatMessageChunk"]`:= Stores the chunk-specific message type used during serialization and deserialization. Its default value is `"ChatMessageChunk"`.
## Methods
1. `__add__`:= Combines the current chat message chunk with another compatible message chunk.
   When two `ChatMessageChunk` objects are combined, their roles must match. Otherwise, a `ValueError` is raised.
   The method merges message content, additional keyword arguments, and response metadata while preserving the current chunk's role and identifier.
   ```python
   __add__(
       self,
       other: Any # Message chunk or another supported value to combine
   ) -> BaseMessageChunk
   ```


In [3]:
from langchain_core.messages import ChatMessageChunk


chunk1 = ChatMessageChunk(
    content="LangChain ", # First partial message content
    role="assistant", # Speaker or role of the message
    name="model", # Optional message name
    id="chunk-101", # Identifier retained in the combined chunk
    additional_kwargs={
        "source": "chat-model" # Provider-specific information
    },
    response_metadata={
        "chunk_number": 1 # Metadata of the first chunk
    }
)

chunk2 = ChatMessageChunk(
    content="supports streaming.", # Second partial message content
    role="assistant", # Must match the first chunk's role
    additional_kwargs={
        "format": "text" # Additional provider-specific information
    },
    response_metadata={
        "completed": True # Metadata of the second chunk
    }
)


combined_chunk = chunk1 + chunk2 # Combine compatible message chunks


print("Content:", combined_chunk.content) # Merged message content
print("Role:", combined_chunk.role) # Preserved message role
print("Type:", combined_chunk.type) # Chunk serialization type
print("Name:", combined_chunk.name) # Preserved optional name
print("ID:", combined_chunk.id) # Preserved identifier
print("Additional kwargs:", combined_chunk.additional_kwargs) # Merged provider data
print("Response metadata:", combined_chunk.response_metadata) # Merged metadata


try:
    invalid_chunk = ChatMessageChunk(
        content="This role is different.",
        role="user" # Role does not match "assistant"
    )

    result = chunk1 + invalid_chunk # Raises ValueError

except ValueError as error:
    print("ValueError:", error)

Content: LangChain supports streaming.
Role: assistant
Type: ChatMessageChunk
Name: None
ID: chunk-101
Additional kwargs: {'source': 'chat-model', 'format': 'text'}
Response metadata: {'chunk_number': 1, 'completed': True}
ValueError: Cannot concatenate ChatMessageChunks with different roles.
